# 05 — Unified Evaluation (YOLOv8 + Faster R-CNN)

Evaluate both models on the same test set using identical metrics:
- mAP @0.5 and @0.5:0.95
- Per-class AP
- Precision, Recall, F1
- Inference Speed (FPS)
- Model Size & Parameters
- Confusion Matrices
- PR Curves
- Qualitative Sample Detections

In [8]:
from pathlib import Path

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "src").is_dir() and (_root / "outputs").is_dir():
        break
else:
    raise FileNotFoundError("Repo root not found (need src/ and outputs/).")

print(f"Repo root: {_root}")

Repo root: C:\Users\micha\Downloads\Object-Detection-main\Object-Detection-main


In [9]:
!pip install ultralytics --no-deps

In [10]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "src").is_dir() and (_root / "outputs").is_dir():
        break
else:
    raise FileNotFoundError("Repo root not found (need src/ and outputs/).")

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import os
import json
import torch
import numpy as np
import pandas as pd
import cv2
import random
import importlib
import src.evaluate
importlib.reload(src.evaluate)

from src.utils import SEED, CLASS_MAP, FRCNN_CLASS_MAP, NUM_CLASSES, CLASS_NAMES, seed_everything
from src.fasterrcnn_dataset import BDD100KDataset
from src.fasterrcnn_utils import build_fasterrcnn, collate_fn
from src.evaluate import (
    predict_yolov8, predict_fasterrcnn,
    compute_map, compute_precision_recall_f1,
    measure_fps, get_model_size_mb, count_all_params,
    plot_pr_curve, build_confusion_matrix,
    draw_boxes, plot_sample_detections, plot_per_class_ap,
    build_pr_curve_data, filter_predictions_by_score,
)

seed_everything(SEED)
print(f"Repo root: {_root}")


Repo root: C:\Users\micha\Downloads\Object-Detection-main\Object-Detection-main


## 1. Load Models

In [11]:
from ultralytics import YOLO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

YOLO_PRETRAINED_PATH = _root / "notebooks" / "yolov8m.pt"
YOLO_TRAINED_PATH = _root / "outputs" / "bdd100k_project" / "runs" / "trained" / "yolov8m_bdd100k_best.pt"

FRCNN_PRETRAINED_PATH = _root / "outputs" / "bdd100k_project" / "fasterrcnn_pretrained.pth"
FRCNN_TRAINED_PATH = _root / "outputs" / "bdd100k_project" / "fasterrcnn_best.pth"

yolo_pretrained = YOLO(str(YOLO_PRETRAINED_PATH))
yolo_trained = YOLO(str(YOLO_TRAINED_PATH))

frcnn_pretrained = build_fasterrcnn(num_classes=NUM_CLASSES)
frcnn_pretrained.load_state_dict(torch.load(FRCNN_PRETRAINED_PATH, map_location=device))
frcnn_pretrained.to(device)
frcnn_pretrained.eval()

frcnn_trained = build_fasterrcnn(num_classes=NUM_CLASSES)
frcnn_trained.load_state_dict(torch.load(FRCNN_TRAINED_PATH, map_location=device))
frcnn_trained.to(device)
frcnn_trained.eval()

print("All pretrained and trained models loaded successfully")

All pretrained and trained models loaded successfully


## 2. Load Test Dataset

In [12]:
DATASET_ROOT = _root / "outputs" / "bdd100k_preprocessing"

test_dataset = BDD100KDataset(
    image_dir=str(DATASET_ROOT / "bdd100k-yolo-subset-v1" / "images" / "test"),
    annotation_file=str(DATASET_ROOT / "test_annotations.json"),
    class_map=FRCNN_CLASS_MAP,
)

print(f"Test samples: {len(test_dataset)}")


Test samples: 1500


## 3. Run Inference on Test Set

In [15]:
from tqdm import tqdm
from torch.utils.data import DataLoader

test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

yolo_pretrained_predictions = []
yolo_trained_predictions = []
frcnn_pretrained_predictions = []
frcnn_trained_predictions = []
all_targets = []

for images, targets in tqdm(test_loader, desc="Running inference"):
    img_tensor = images[0]
    target = targets[0]

    image_id = target["image_id"].item()
    item = test_dataset.annotations[image_id]

    image_name = item["name"]
    if not os.path.splitext(image_name)[1]:
        image_name = f"{image_name}.jpg"

    img_path = os.path.join(test_dataset.image_dir, image_name)

    yolo_pre_preds = predict_yolov8(yolo_pretrained, img_path)
    yolo_tr_preds = predict_yolov8(yolo_trained, img_path)

    frcnn_pre_preds = predict_fasterrcnn(frcnn_pretrained, img_tensor, device)
    frcnn_tr_preds = predict_fasterrcnn(frcnn_trained, img_tensor, device)

    def to_prediction_dict(preds):
        return {
            "boxes": torch.tensor([p["box"] for p in preds], dtype=torch.float32) if preds else torch.zeros((0, 4)),
            "scores": torch.tensor([p["score"] for p in preds]) if preds else torch.zeros(0),
            "labels": torch.tensor([p["label"] for p in preds], dtype=torch.int64) if preds else torch.zeros(0, dtype=torch.int64),
        }

    yolo_pretrained_predictions.append(to_prediction_dict(yolo_pre_preds))
    yolo_trained_predictions.append(to_prediction_dict(yolo_tr_preds))
    frcnn_pretrained_predictions.append(to_prediction_dict(frcnn_pre_preds))
    frcnn_trained_predictions.append(to_prediction_dict(frcnn_tr_preds))

    all_targets.append({
        "boxes": target["boxes"],
        "labels": target["labels"]
    })

print(f"Inference complete: {len(all_targets)} images")

Running inference: 100%|██████████| 1500/1500 [01:38<00:00, 15.25it/s]

Inference complete: 1500 images


## 4. mAP Computation

In [16]:
!{sys.executable} -m pip install faster-coco-eval

In [17]:
print("Computing mAP for YOLO pretrained...")
yolo_pre_map = compute_map(yolo_pretrained_predictions, all_targets)

print("Computing mAP for YOLO trained...")
yolo_tr_map = compute_map(yolo_trained_predictions, all_targets)

print("Computing mAP for Faster R-CNN pretrained...")
frcnn_pre_map = compute_map(frcnn_pretrained_predictions, all_targets)

print("Computing mAP for Faster R-CNN trained...")
frcnn_tr_map = compute_map(frcnn_trained_predictions, all_targets)


def print_map_results(name, result):
    print(f"\n--- {name} ---")
    print(f"mAP@0.5:0.95: {result['map'].item():.4f}")
    print(f"mAP@0.5:      {result['map_50'].item():.4f}")
    print(f"mAP@0.75:     {result['map_75'].item():.4f}")
    print(f"mAP (small):  {result['map_small'].item():.4f}")
    print(f"mAP (medium): {result['map_medium'].item():.4f}")
    print(f"mAP (large):  {result['map_large'].item():.4f}")


print_map_results("YOLOv8 Pretrained", yolo_pre_map)
print_map_results("YOLOv8 Trained", yolo_tr_map)
print_map_results("Faster R-CNN Pretrained", frcnn_pre_map)
print_map_results("Faster R-CNN Trained", frcnn_tr_map)

Computing mAP for YOLO pretrained...
Computing mAP for YOLO trained...
Computing mAP for Faster R-CNN pretrained...
Computing mAP for Faster R-CNN trained...

--- YOLOv8 Pretrained ---
mAP@0.5:0.95: 0.0001
mAP@0.5:      0.0002
mAP@0.75:     0.0001
mAP (small):  0.0001
mAP (medium): 0.0001
mAP (large):  0.0001

--- YOLOv8 Trained ---
mAP@0.5:0.95: 0.2593
mAP@0.5:      0.4536
mAP@0.75:     0.2400
mAP (small):  0.1019
mAP (medium): 0.3141
mAP (large):  0.5339

--- Faster R-CNN Pretrained ---
mAP@0.5:0.95: 0.0001
mAP@0.5:      0.0003
mAP@0.75:     0.0000
mAP (small):  0.0000
mAP (medium): 0.0001
mAP (large):  0.0000

--- Faster R-CNN Trained ---
mAP@0.5:0.95: 0.2138
mAP@0.5:      0.4207
mAP@0.75:     0.1834
mAP (small):  0.0948
mAP (medium): 0.2711
mAP (large):  0.4462


In [18]:
summary_df = pd.DataFrame([
    {"Model": "YOLOv8 Pretrained", "mAP@0.5:0.95": yolo_pre_map["map"].item(), "mAP@0.5": yolo_pre_map["map_50"].item(), "mAP@0.75": yolo_pre_map["map_75"].item()},
    {"Model": "YOLOv8 Trained", "mAP@0.5:0.95": yolo_tr_map["map"].item(), "mAP@0.5": yolo_tr_map["map_50"].item(), "mAP@0.75": yolo_tr_map["map_75"].item()},
    {"Model": "Faster R-CNN Pretrained", "mAP@0.5:0.95": frcnn_pre_map["map"].item(), "mAP@0.5": frcnn_pre_map["map_50"].item(), "mAP@0.75": frcnn_pre_map["map_75"].item()},
    {"Model": "Faster R-CNN Trained", "mAP@0.5:0.95": frcnn_tr_map["map"].item(), "mAP@0.5": frcnn_tr_map["map_50"].item(), "mAP@0.75": frcnn_tr_map["map_75"].item()},
])

summary_df

,Model,mAP@0.5:0.95,mAP@0.5,mAP@0.75
0,YOLOv8 Pretrained,0.000092,0.000197,0.000075
1,YOLOv8 Trained,0.259290,0.453629,0.240020
2,Faster R-CNN Pretrained,0.000062,0.000309,0.000000
3,Faster R-CNN Trained,0.213785,0.420712,0.183431


## 5. Per-Class AP

In [19]:
CLASS_NAMES_LIST = [CLASS_NAMES[i] for i in range(len(CLASS_MAP))]

yolo_per_class = yolo_tr_map["map_per_class"].numpy()
frcnn_per_class = frcnn_tr_map["map_per_class"].numpy()

per_class_df = pd.DataFrame({
    "Class": CLASS_NAMES_LIST,
    "YOLOv8 Trained AP": yolo_per_class,
    "Faster R-CNN Trained AP": frcnn_per_class,
})
print(per_class_df.to_string(index=False))

plot_per_class_ap(
    yolo_per_class,
    frcnn_per_class,
    CLASS_NAMES_LIST,
    save_path=str(_root / "outputs" / "bdd100k_project" / "results" / "plots" / "per_class_ap_comparison.png")
)

        Class  YOLOv8 Trained AP  Faster R-CNN Trained AP
          car           0.421897                 0.396715
       person           0.263055                 0.274346
        truck           0.330475                 0.254162
          bus           0.347106                 0.153805
        motor           0.108033                 0.056999
         bike           0.129228                 0.125512
traffic light           0.181896                 0.170190
 traffic sign           0.292634                 0.278551
Per-class AP chart saved to C:\Users\micha\Downloads\Object-Detection-main\Object-Detection-main\outputs\bdd100k_project\results\plots\per_class_ap_comparison.png


## 6. Precision, Recall, F1

In [20]:
yolo_pre_precision, yolo_pre_recall, yolo_pre_f1 = compute_precision_recall_f1(
    yolo_pretrained_predictions, all_targets
)
yolo_tr_precision, yolo_tr_recall, yolo_tr_f1 = compute_precision_recall_f1(
    yolo_trained_predictions, all_targets
)

frcnn_pre_precision, frcnn_pre_recall, frcnn_pre_f1 = compute_precision_recall_f1(
    frcnn_pretrained_predictions, all_targets
)
frcnn_tr_precision, frcnn_tr_recall, frcnn_tr_f1 = compute_precision_recall_f1(
    frcnn_trained_predictions, all_targets
)

print(f"YOLOv8 Pretrained       — Precision: {yolo_pre_precision:.4f} | Recall: {yolo_pre_recall:.4f} | F1: {yolo_pre_f1:.4f}")
print(f"YOLOv8 Trained          — Precision: {yolo_tr_precision:.4f} | Recall: {yolo_tr_recall:.4f} | F1: {yolo_tr_f1:.4f}")
print(f"Faster R-CNN Pretrained — Precision: {frcnn_pre_precision:.4f} | Recall: {frcnn_pre_recall:.4f} | F1: {frcnn_pre_f1:.4f}")
print(f"Faster R-CNN Trained    — Precision: {frcnn_tr_precision:.4f} | Recall: {frcnn_tr_recall:.4f} | F1: {frcnn_tr_f1:.4f}")

YOLOv8 Pretrained       — Precision: 0.0066 | Recall: 0.0031 | F1: 0.0042
YOLOv8 Trained          — Precision: 0.7415 | Recall: 0.6525 | F1: 0.6942
Faster R-CNN Pretrained — Precision: 0.0769 | Recall: 0.0000 | F1: 0.0001
Faster R-CNN Trained    — Precision: 0.6246 | Recall: 0.6808 | F1: 0.6515


## 7. Inference Speed (FPS)

In [21]:
FPS_SAMPLE_SIZE = 200
random.seed(SEED)

fps_indices = random.sample(range(len(test_dataset)), min(FPS_SAMPLE_SIZE, len(test_dataset)))
fps_items = [test_dataset[i] for i in fps_indices]
fps_image_tensors = [item[0] for item in fps_items]

fps_image_paths = []
for idx in fps_indices:
    item = test_dataset.annotations[idx]
    image_name = item["name"]
    if not os.path.splitext(image_name)[1]:
        image_name = f"{image_name}.jpg"
    fps_image_paths.append(os.path.join(test_dataset.image_dir, image_name))

yolo_fps, yolo_ms = measure_fps(
    lambda img: predict_yolov8(yolo_trained, img),
    fps_image_paths,
    n_warmup=10
)

frcnn_fps, frcnn_ms = measure_fps(
    lambda img: predict_fasterrcnn(frcnn_trained, img, device),
    fps_image_tensors,
    n_warmup=10
)

print(f"YOLOv8 Trained       — FPS: {yolo_fps:.1f} | Latency: {yolo_ms:.1f} ms/image")
print(f"Faster R-CNN Trained — FPS: {frcnn_fps:.1f} | Latency: {frcnn_ms:.1f} ms/image")

YOLOv8 Trained       — FPS: 107.6 | Latency: 9.3 ms/image
Faster R-CNN Trained — FPS: 45.3 | Latency: 22.1 ms/image


## 8. Model Size & Parameters

In [22]:
yolo_size = get_model_size_mb(str(YOLO_TRAINED_PATH))
frcnn_size = get_model_size_mb(str(FRCNN_TRAINED_PATH))

yolo_params = count_all_params(yolo_trained.model)
frcnn_params = count_all_params(frcnn_trained)

print(f"YOLOv8 Trained       — Size: {yolo_size:.1f} MB | Parameters: {yolo_params:,}")
print(f"Faster R-CNN Trained — Size: {frcnn_size:.1f} MB | Parameters: {frcnn_params:,}")

YOLOv8 Trained       — Size: 49.6 MB | Parameters: 25,844,392
Faster R-CNN Trained — Size: 158.2 MB | Parameters: 41,335,036


## 9. Confusion Matrices

In [23]:
build_confusion_matrix(
    yolo_trained_predictions,
    all_targets,
    CLASS_NAMES_LIST,
    iou_threshold=0.5,
    model_name="YOLOv8 Trained",
    save_path=str(_root / "outputs" / "bdd100k_project" / "results" / "plots" / "confusion_matrix_yolov8_trained.png")
)

build_confusion_matrix(
    frcnn_trained_predictions,
    all_targets,
    CLASS_NAMES_LIST,
    iou_threshold=0.5,
    model_name="Faster R-CNN Trained",
    save_path=str(_root / "outputs" / "bdd100k_project" / "results" / "plots" / "confusion_matrix_fasterrcnn_trained.png")
)


Confusion matrix saved to C:\Users\micha\Downloads\Object-Detection-main\Object-Detection-main\outputs\bdd100k_project\results\plots\confusion_matrix_yolov8_trained.png
Confusion matrix saved to C:\Users\micha\Downloads\Object-Detection-main\Object-Detection-main\outputs\bdd100k_project\results\plots\confusion_matrix_fasterrcnn_trained.png


array([[11310,     4,    39,     4,     0,     1,     7,    11,  4415],
       [    9,  1169,     0,     0,     0,     4,     0,     5,   712],
       [  168,     1,   278,     7,     0,     0,     0,    13,   183],
       [   53,     0,    49,    55,     0,     0,     0,     3,    84],
       [    8,     3,     0,     0,     9,     4,     0,     0,    49],
       [    3,     7,     0,     0,     0,    56,     0,     0,    83],
       [    0,     0,     0,     0,     0,     0,  2585,    56,  1404],
       [    9,     0,     0,     0,     0,     0,    45,  3535,  1531],
       [ 3691,   854,   128,    15,     5,    78,  2131,  4075,     0]])

## 10. PR Curves

In [24]:
yolo_precisions, yolo_recalls = build_pr_curve_data(yolo_trained_predictions, all_targets)
frcnn_precisions, frcnn_recalls = build_pr_curve_data(frcnn_trained_predictions, all_targets)

plot_pr_curve(
    yolo_precisions,
    yolo_recalls,
    "YOLOv8 Trained",
    str(_root / "outputs" / "bdd100k_project" / "results" / "plots" / "pr_curve_yolov8_trained.png")
)

plot_pr_curve(
    frcnn_precisions,
    frcnn_recalls,
    "Faster R-CNN Trained",
    str(_root / "outputs" / "bdd100k_project" / "results" / "plots" / "pr_curve_fasterrcnn_trained.png")
)

PR curve saved to C:\Users\micha\Downloads\Object-Detection-main\Object-Detection-main\outputs\bdd100k_project\results\plots\pr_curve_yolov8_trained.png
PR curve saved to C:\Users\micha\Downloads\Object-Detection-main\Object-Detection-main\outputs\bdd100k_project\results\plots\pr_curve_fasterrcnn_trained.png


## 11. Qualitative Sample Detections

In [25]:
random.seed(SEED)
N_SAMPLES = 6
sample_indices = random.sample(range(len(test_dataset)), N_SAMPLES)

test_items = []
for idx in sample_indices:
    img_tensor, target = test_dataset[idx]
    item = test_dataset.annotations[idx]

    image_name = item["name"]
    if not os.path.splitext(image_name)[1]:
        image_name = f"{image_name}.jpg"

    img_path = os.path.join(test_dataset.image_dir, image_name)

    gt_boxes = target["boxes"].numpy().tolist()
    gt_labels = target["labels"].numpy().tolist()

    test_items.append({
        "image_path": img_path,
        "gt_boxes": gt_boxes,
        "gt_labels": gt_labels,
    })

plot_sample_detections(
    test_items,
    yolo_trained,
    frcnn_trained,
    CLASS_NAMES_LIST,
    device,
    n=N_SAMPLES,
    save_path=str(_root / "outputs" / "bdd100k_project" / "results" / "plots" / "qualitative_samples.png")
)

Sample detections saved to C:\Users\micha\Downloads\Object-Detection-main\Object-Detection-main\outputs\bdd100k_project\results\plots\qualitative_samples.png


## 12. Comparison Table

In [26]:
with open(_root / "outputs" / "bdd100k_project" / "fasterrcnn_loss_history.json") as f:
    frcnn_history = json.load(f)

frcnn_train_hours = frcnn_history.get("training_time_hours", 0)

comparison = pd.DataFrame({
    "Metric": [
        "mAP@0.5",
        "mAP@0.5:0.95",
        "mAP@0.75",
        "mAP (small objects)",
        "mAP (medium objects)",
        "mAP (large objects)",
        "Precision",
        "Recall",
        "F1 Score",
        "Inference Speed (FPS)",
        "Latency (ms/image)",
        "Model Size (MB)",
        "Parameters (M)",
        "Training Time (hrs)",
    ],
    "YOLOv8m Trained": [
        f"{yolo_tr_map['map_50'].item():.4f}",
        f"{yolo_tr_map['map'].item():.4f}",
        f"{yolo_tr_map['map_75'].item():.4f}",
        f"{yolo_tr_map['map_small'].item():.4f}",
        f"{yolo_tr_map['map_medium'].item():.4f}",
        f"{yolo_tr_map['map_large'].item():.4f}",
        f"{yolo_tr_precision:.4f}",
        f"{yolo_tr_recall:.4f}",
        f"{yolo_tr_f1:.4f}",
        f"{yolo_fps:.1f}",
        f"{yolo_ms:.1f}",
        f"{yolo_size:.1f}",
        f"{yolo_params / 1e6:.1f}",
        "—",
    ],
    "Faster R-CNN Trained": [
        f"{frcnn_tr_map['map_50'].item():.4f}",
        f"{frcnn_tr_map['map'].item():.4f}",
        f"{frcnn_tr_map['map_75'].item():.4f}",
        f"{frcnn_tr_map['map_small'].item():.4f}",
        f"{frcnn_tr_map['map_medium'].item():.4f}",
        f"{frcnn_tr_map['map_large'].item():.4f}",
        f"{frcnn_tr_precision:.4f}",
        f"{frcnn_tr_recall:.4f}",
        f"{frcnn_tr_f1:.4f}",
        f"{frcnn_fps:.1f}",
        f"{frcnn_ms:.1f}",
        f"{frcnn_size:.1f}",
        f"{frcnn_params / 1e6:.1f}",
        f"{frcnn_train_hours:.2f}",
    ],
})

print(comparison.to_string(index=False))

comparison_path = _root / "outputs" / "bdd100k_project" / "results" / "metrics" / "comparison_table.csv"
comparison.to_csv(comparison_path, index=False)

print(f"\nComparison table saved to {comparison_path}")

               Metric YOLOv8m Trained Faster R-CNN Trained
              mAP@0.5          0.4536               0.4207
         mAP@0.5:0.95          0.2593               0.2138
             mAP@0.75          0.2400               0.1834
  mAP (small objects)          0.1019               0.0948
 mAP (medium objects)          0.3141               0.2711
  mAP (large objects)          0.5339               0.4462
            Precision          0.7415               0.6246
               Recall          0.6525               0.6808
             F1 Score          0.6942               0.6515
Inference Speed (FPS)           107.6                 45.3
   Latency (ms/image)             9.3                 22.1
      Model Size (MB)            49.6                158.2
       Parameters (M)            25.8                 41.3
  Training Time (hrs)               —                 0.67

Comparison table saved to C:\Users\micha\Downloads\Object-Detection-main\Object-Detection-main\outputs\bdd100k_pro